# Phase 1 — Baseline Replication on Colab
**Small Models, Honest Maps** — Uncertainty-Aware Lightweight Flood Segmentation from Sentinel-1

This notebook:
1. Clones the project repo and installs dependencies
2. Downloads the ETCI 2021 dataset (5.6 GB, Hugging Face mirror)
3. Runs a 2-epoch **smoke run** to verify the full pipeline
4. Launches the **full baseline training** (UNet++ + EfficientNet-B7, 100 epochs)
5. Saves checkpoints to **Google Drive** after every improvement

**Before you start:** Runtime → Change runtime type → GPU (A100/T4).

## 1. Setup — clone repo and install dependencies

In [ ]:
import os

REPO = "https://github.com/nazizahed/Uncertainty-Aware-Flood-Segmentation-from-Sentinel-1-for-Near-Real-Time-Applications.git"
WORKDIR = "/content/sar-flood-uq"

if not os.path.exists(WORKDIR):
    !git clone {REPO} {WORKDIR}
%cd {WORKDIR}
!pip install -q -r requirements.txt
print("setup done")

## 2. Mount Google Drive (checkpoint persistence)

Colab sessions disconnect. Checkpoints are mirrored to Drive on every improvement.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = "/content/drive/MyDrive/sar-flood-uq"
os.makedirs(DRIVE_DIR, exist_ok=True)
print("checkpoints will be mirrored to:", DRIVE_DIR)

## 3. Download the ETCI 2021 dataset (~5.6 GB)

Community mirror of the NASA IMPACT / IEEE GRSS competition data. Register at
https://nasa-impact.github.io/etci2021/ as well (publication acknowledgement requirement).

In [ ]:
!python scripts/download_data.py --out data/etci

In [ ]:
# sanity check: what did we get?
from pathlib import Path
root = Path("data/etci")
region_dirs = sorted(p for p in root.rglob("*") if p.is_dir() and (p / "vv").is_dir())
print(f"{len(region_dirs)} region folders found:")
for p in region_dirs:
    print("  ", p.relative_to(root))

## 4. Verify the data pipeline

Loads the dataset through the real loader: ratio channel, stratified batches, augmentation.

In [ ]:
import sys
sys.path.insert(0, "src")
import torch, yaml
from sarflood.data.dataset import ETCIFloodDataset

with open("configs/baseline_unetpp_b7.yaml") as f:
    cfg = yaml.safe_load(f)

train_ds = ETCIFloodDataset(cfg["data"]["root"], cfg["data"]["regions"], cfg["data"]["bands"], rotation_aug=True)
val_ds   = ETCIFloodDataset(cfg["data"]["root"], cfg["data"]["val_regions"], cfg["data"]["bands"])
print(f"train tiles: {len(train_ds)}  (expect ~25,000)")
print(f"val tiles (florence): {len(val_ds)}  (expect ~8,382)")
print(f"flood-positive train tiles: {(train_ds.flood_fraction > 0).mean():.1%}")
item = train_ds[0]
print("tile:", item["image"].shape, "| mask:", item["mask"].shape, "| region:", item["region"])

## 5. Smoke run — 2 epochs, small model

Verifies the training loop end-to-end before committing GPU hours to the real run.

In [ ]:
import copy
smoke = copy.deepcopy(cfg)
smoke["experiment_name"] = "smoke_test"
smoke["model"] = {"arch": "unet", "encoder": "resnet18", "encoder_weights": "imagenet", "dropout": 0.2}
smoke["training"]["epochs"] = 2
smoke["training"]["batch_size"] = 32

with open("/tmp/smoke.yaml", "w") as f:
    yaml.safe_dump(smoke, f)

!python scripts/train.py --config /tmp/smoke.yaml

## 6. Full baseline training — UNet++ + EfficientNet-B7, 100 epochs

**This is the Phase 1 anchor run.** On an A100 expect roughly 1.5–3 min/epoch with AMP (~2.5–5 h total);
on a T4 considerably longer. Target: Florence val IoU ≈ 75.8 (Ghosh et al. 2024, Table 2).

In [ ]:
import shutil, time, threading, os

def mirror_to_drive(run_dir, drive_dir, stop):
    """Background thread: copy checkpoints to Drive every 5 minutes."""
    while not stop.is_set():
        os.makedirs(drive_dir, exist_ok=True)
        for f in ["best.pt", "last.pt", "log.csv", "config.yaml"]:
            src = os.path.join(run_dir, f)
            if os.path.exists(src):
                shutil.copy2(src, drive_dir)
        stop.wait(300)

run_dir = os.path.join(WORKDIR, "runs", cfg["experiment_name"])
drive_run = os.path.join(DRIVE_DIR, cfg["experiment_name"])
stop = threading.Event()
t = threading.Thread(target=mirror_to_drive, args=(run_dir, drive_run, stop), daemon=True)
t.start()

!python scripts/train.py --config configs/baseline_unetpp_b7.yaml

stop.set(); t.join()
mirror_to_drive(run_dir, drive_run, threading.Event())  # final copy
print("done. checkpoints at:", drive_run)

## 7. Evaluate on Florence validation

Compare against Ghosh et al. (2024) Table 2: Acc 98.8 / Prec 89.5 / Rec 89.1 / F1 89.3 / IoU 75.76 / Kappa 81.6.
Replication target: IoU within ~1–2 points.

In [ ]:
!python scripts/evaluate.py     --checkpoint runs/baseline_unetpp_efficientnetb7/best.pt     --regions florence     --out runs/baseline_unetpp_efficientnetb7/val_results.json

## 8. (Optional) Uncertainty preview — MC Dropout on validation tiles

20 stochastic passes → predictive mean + entropy maps, plus calibration (ECE/Brier)
and risk–coverage (AURC). Full UQ analysis is Phase 3; this is a sanity preview.

In [ ]:
!python scripts/evaluate.py     --checkpoint runs/baseline_unetpp_efficientnetb7/best.pt     --regions florence --mc-passes 20 --batch-size 8     --out runs/baseline_unetpp_efficientnetb7/val_results_mc20.json